# E1. Дообучение RuBERT в Google Colab

1. Загрузите этот ноутбук в Google Colab и выберите среду с GPU, например T4.
2. Запустите ячейки сверху вниз. В ячейке загрузки выберите `e1_rubert_data.zip` или одновременно три файла `train.jsonl`, `validation.jsonl` и `test.jsonl` из `data/synthetic/`.
3. Скачайте результаты из последней ячейки. Обученную модель и токенизатор можно скачать отдельно.

Ноутбук использует синтетический корпус лабораторной. Веса RuBERT загружаются из Hugging Face, обучение и оценка выполняются средствами PyTorch в среде Colab.

In [ ]:
%pip install -q "transformers==5.0.0" "torch>=2.6"

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import hashlib
import io
import json
import platform
import re
import shutil
import tempfile
import zipfile
from collections import Counter
from pathlib import Path

import tokenizers
import torch
import transformers
from google.colab import files
from torch.utils.data import DataLoader
from transformers import (
    AutoModelForTokenClassification, AutoTokenizer,
    DataCollatorForTokenClassification, set_seed,
)

## 1. Параметры

Дообучаются все веса RuBERT и линейный классификатор на 17 BIO-меток. Контрольная точка и токенизатор загружаются из одной фиксированной ревизии. Выбирается эпоха с максимальным микро-F1 по сущностям на `validation`. При равенстве выбирается более ранняя эпоха.

[Карточка RuBERT](https://huggingface.co/DeepPavlov/rubert-base-cased) · [Ревизия](https://huggingface.co/DeepPavlov/rubert-base-cased/tree/4036cab694767a299f2b9e6492909664d9414229)

In [ ]:
config = {
    "model_id": "DeepPavlov/rubert-base-cased",
    "revision": "4036cab694767a299f2b9e6492909664d9414229",
    "seed": 42,
    "epochs": 6,
    "batch_size": 4,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "max_length": 256,
    "max_grad_norm": 1.0,
    "num_threads": 4,
}

torch.set_num_threads(config["num_threads"])
torch.use_deterministic_algorithms(True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU"
print("Устройство:", device_name)

## 2. Загрузка и проверка корпуса

ZIP читается в памяти. Проверяются символьные позиции, BIO-разметка и отсутствие общих шаблонов, идентификаторов или текстов между выборками. Тестовая выборка используется для оценки только после выбора лучшей эпохи.

In [ ]:
LABELS = (
    "NAME", "DATE", "PHONE", "EMAIL", "ID", "LOCATION", "ORGANIZATION", "AGE"
)
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)


def validate_record(record):
    text = record["text"]
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Пустой текст")
    previous_end = 0
    for entity in record["entities"]:
        start, end = entity["start"], entity["end"]
        if type(start) is not int or type(end) is not int:
            raise ValueError("Позиции должны быть целыми")
        if not 0 <= start < end <= len(text):
            raise ValueError("Диапазон вне текста")
        if start < previous_end:
            raise ValueError("Пересекающиеся или неупорядоченные сущности")
        if entity["label"] not in LABELS:
            raise ValueError("Неизвестный класс")
        if text[start:end] != entity["text"]:
            raise ValueError("Неверная подстрока сущности")
        if entity["text"] != entity["text"].strip():
            raise ValueError("Пробелы на границах")
        previous_end = end

    tokens, tags = record["tokens"], record["bio_tags"]
    matches = list(TOKEN_PATTERN.finditer(text))
    if not len(tokens) == len(tags) == len(matches):
        raise ValueError("Число токенов и меток различается")
    for token, match in zip(tokens, matches):
        if token != {
            "text": match.group(), "start": match.start(), "end": match.end()
        }:
            raise ValueError("Неверные позиции токена")

    recovered = []
    for token, tag in zip(tokens, tags):
        if tag == "O":
            continue
        if tag[:2] not in ("B-", "I-") or tag[2:] not in LABELS:
            raise ValueError("Неверная BIO-метка")
        if tag.startswith("B-"):
            recovered.append({
                "start": token["start"], "end": token["end"], "label": tag[2:]
            })
        else:
            if not recovered:
                raise ValueError("I-метка без начала сущности")
            entity = recovered[-1]
            if entity["label"] != tag[2:]:
                raise ValueError("Смена типа внутри сущности")
            gap = text[entity["end"]:token["start"]]
            if gap and not gap.isspace():
                raise ValueError("Разрыв внутри BIO-сущности")
            entity["end"] = token["end"]
    expected = [
        {key: entity[key] for key in ("start", "end", "label")}
        for entity in record["entities"]
    ]
    if recovered != expected:
        raise ValueError("BIO-разметка не восстанавливает сущности")


def validate_corpus(corpus):
    if set(corpus) != {"train", "validation", "test"}:
        raise ValueError("Неверные части корпуса")
    seen_ids, seen_texts, seen_templates = set(), set(), set()
    totals = Counter()
    positives = negatives = 0
    for split, records in corpus.items():
        if not records:
            raise ValueError("Пустая выборка")
        template_ids = set()
        counts = Counter()
        negative_count = 0
        for record in records:
            validate_record(record)
            if record["split"] != split:
                raise ValueError("Неверная принадлежность выборке")
            if record["id"] in seen_ids:
                raise ValueError("Повтор идентификатора")
            if record["text"] in seen_texts:
                raise ValueError("Дубликат текста")
            seen_ids.add(record["id"])
            seen_texts.add(record["text"])
            template_ids.add(record["template_id"])
            counts.update(entity["label"] for entity in record["entities"])
            if record["entities"]:
                positives += 1
            else:
                negatives += 1
                negative_count += 1
        if template_ids & seen_templates:
            raise ValueError("Утечка шаблонов между выборками")
        seen_templates.update(template_ids)
        if any(counts[label] == 0 for label in LABELS):
            raise ValueError("В выборке отсутствует класс")
        if negative_count == 0:
            raise ValueError("Нет отрицательных примеров")
        totals.update(counts)
    if positives < 120:
        raise ValueError("Недостаточно записей с сущностями")
    if negatives < 30:
        raise ValueError("Недостаточно отрицательных записей")
    if any(totals[label] < 25 for label in LABELS):
        raise ValueError("Недостаточно сущностей класса")
    return totals


def load_corpus(uploaded):
    required = {"train.jsonl", "validation.jsonl", "test.jsonl"}
    if len(uploaded) == 1 and next(iter(uploaded)).lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
            names = archive.namelist()
            if len(names) != len(required) or set(names) != required:
                raise ValueError("В ZIP нужны train.jsonl, validation.jsonl и test.jsonl без подпапок")
            uploaded = {name: archive.read(name) for name in names}
    if set(uploaded) != required:
        raise ValueError("Загрузите e1_rubert_data.zip или все три файла JSONL")

    corpus = {
        split: [
            json.loads(line)
            for line in uploaded[f"{split}.jsonl"].decode("utf-8").splitlines()
        ]
        for split in ("train", "validation", "test")
    }
    validate_corpus(corpus)
    hashes = {
        name: hashlib.sha256(content).hexdigest()
        for name, content in sorted(uploaded.items())
    }
    return corpus, hashes

In [ ]:
corpus, input_hashes = load_corpus(files.upload())
for split, records in corpus.items():
    print(f"{split}: {len(records)} записей")

## 3. Согласование BIO и метрики

Первый подтокен получает исходную BIO-метку, продолжение сущности получает I-метку, продолжение обычного слова — O. Специальные токены и дополнение получают `-100`. Смещения подтокенов переводятся в позиции исходного текста. При превышении `max_length` запись отклоняется целиком.

Сущность считается найденной при совпадении класса, начала и конца. Сохраняются TP, FP, FN, precision, recall и F1 по классам, микро- и макроусреднение и дополнительная точность по активным подтокенам.

In [ ]:
BIO_LABELS = ("O",) + tuple(prefix + label for label in LABELS for prefix in ("B-", "I-"))
LABEL2ID = {label: index for index, label in enumerate(BIO_LABELS)}
ID2LABEL = {index: label for label, index in LABEL2ID.items()}


def align_labels(word_labels, word_ids):
    """Последующие подтокены сущности получают I-метку, служебные — -100."""
    result = []
    previous = None
    for word_id in word_ids:
        if word_id is None:
            result.append(-100)
        else:
            if type(word_id) is not int or not 0 <= word_id < len(word_labels):
                raise ValueError("Подтокен ссылается на отсутствующее слово")
            label = word_labels[word_id]
            if word_id == previous and label.startswith("B-"):
                label = "I-" + label[2:]
            if label not in LABEL2ID:
                raise ValueError("Неизвестная BIO-метка: " + label)
            result.append(LABEL2ID[label])
        previous = word_id
    return result


def encode_record(record, tokenizer, max_length):
    """Подготовка входов и абсолютных позиций без скрытого обрезания текста."""
    validate_record(record)
    if not tokenizer.is_fast:
        raise ValueError("Требуется быстрый токенизатор с word_ids()")
    words = [token["text"] for token in record["tokens"]]
    encoded = tokenizer(
        words, is_split_into_words=True, return_offsets_mapping=True,
        truncation=False, padding=False,
    )
    if len(encoded["input_ids"]) > max_length:
        raise ValueError("Запись {} превышает max_length={}".format(
            record.get("id", "без идентификатора"), max_length
        ))
    word_ids = encoded.word_ids()
    if {word_id for word_id in word_ids if word_id is not None} != set(range(len(words))):
        raise ValueError("Токенизатор пропустил исходные слова")
    offsets = encoded.pop("offset_mapping")
    spans = []
    for word_id, (start, end) in zip(word_ids, offsets):
        if word_id is None:
            spans.append(None)
            continue
        if not 0 <= start < end <= len(words[word_id]):
            raise ValueError("Некорректные позиции подтокена")
        word_start = record["tokens"][word_id]["start"]
        spans.append((word_start + start, word_start + end))
    features = dict(encoded)
    features["labels"] = align_labels(record["bio_tags"], word_ids)
    return features, spans


def decode_entities(predictions, spans):
    """Восстановление сущностей; одиночная I-метка считается началом сущности."""
    if len(predictions) != len(spans):
        raise ValueError("Число предсказаний и диапазонов различается")
    entities = []
    current = None
    previous_end = 0
    for prediction, span in zip(predictions, spans):
        if span is None:
            current = None
            continue
        if prediction not in ID2LABEL:
            raise ValueError("Неизвестный номер BIO-метки")
        start, end = span
        if start < previous_end or start >= end:
            raise ValueError("Диапазоны подтокенов пересекаются или пусты")
        previous_end = end
        tag = ID2LABEL[prediction]
        if tag == "O":
            current = None
            continue
        label = tag[2:]
        if tag.startswith("I-") and current is not None and current["label"] == label:
            current["end"] = end
        else:
            current = {"start": start, "end": end, "label": label}
            entities.append(current)
    return entities

In [ ]:
def entity_keys(entities, text):
    """Проверка диапазонов и преобразование сущностей для строгого сравнения."""
    if not isinstance(entities, list):
        raise ValueError("Сущности должны быть списком")
    keys = set()
    for entity in entities:
        if not isinstance(entity, dict):
            raise ValueError("Сущность должна быть объектом")
        start, end, label = (entity.get(key) for key in ("start", "end", "label"))
        if type(start) is not int or type(end) is not int:
            raise ValueError("Позиции сущности должны быть целыми числами")
        if not 0 <= start < end <= len(text) or label not in LABELS:
            raise ValueError("Недопустимый диапазон или класс сущности")
        key = (start, end, label)
        if key in keys:
            raise ValueError("Повтор сущности с теми же границами и типом")
        keys.add(key)
    return keys


def count_entities(expected, predicted, text, labels=LABELS):
    """TP требует точного совпадения типа, начала и конца внутри одной записи."""
    expected_keys = entity_keys(expected, text)
    predicted_keys = entity_keys(predicted, text)
    matched = expected_keys & predicted_keys
    false_positive = predicted_keys - expected_keys
    false_negative = expected_keys - predicted_keys
    return {
        label: {
            "tp": sum(key[2] == label for key in matched),
            "fp": sum(key[2] == label for key in false_positive),
            "fn": sum(key[2] == label for key in false_negative),
        }
        for label in labels
    }


def calculate_metrics(counts):
    """Расчёт метрик; при нулевом знаменателе результат принят равным нулю."""
    tp, fp, fn = (counts[key] for key in ("tp", "fp", "fn"))
    return dict(
        counts,
        precision=tp / (tp + fp) if tp + fp else 0.0,
        recall=tp / (tp + fn) if tp + fn else 0.0,
        f1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
    )


def summarize(counts, labels):
    micro_counts = {
        key: sum(counts[label][key] for label in labels)
        for key in ("tp", "fp", "fn")
    }
    macro = {
        key: sum(calculate_metrics(counts[label])[key] for label in labels) / len(labels)
        for key in ("precision", "recall", "f1")
    }
    return {"micro": calculate_metrics(micro_counts), "macro": macro}


def evaluate_predictions(records, predictions, labels=LABELS):
    """Строгая оценка списков предсказаний в порядке исходных записей."""
    if len(records) != len(predictions):
        raise ValueError("Число предсказаний не совпадает с числом записей")
    counts = {label: {"tp": 0, "fp": 0, "fn": 0} for label in labels}
    for record, predicted in zip(records, predictions):
        if not isinstance(record, dict) or not isinstance(record.get("text"), str):
            raise ValueError("Запись должна содержать строковое поле text")
        current = count_entities(record.get("entities"), predicted, record["text"], labels)
        for label in labels:
            for key in ("tp", "fp", "fn"):
                counts[label][key] += current[label][key]
    return {
        "records": len(records),
        "per_class": {label: calculate_metrics(counts[label]) for label in labels},
        **summarize(counts, labels),
    }

## 4. Дообучение

После каждой эпохи модель оценивается на `validation`. На диск сохраняется только контрольная точка лучшей эпохи. Журнал содержит средний loss по активным подтокенам и микро-F1 на `validation`.

In [ ]:
def make_loader(features, tokenizer, config, shuffle=False):
    return DataLoader(
        features,
        batch_size=config["batch_size"],
        shuffle=shuffle,
        generator=torch.Generator().manual_seed(config["seed"]),
        collate_fn=DataCollatorForTokenClassification(
            tokenizer=tokenizer, label_pad_token_id=-100,
        ),
    )


def evaluate_model(model, loader, records, spans, device):
    model.eval()
    predictions = []
    correct = total = 0
    with torch.inference_mode():
        for batch in loader:
            targets = batch["labels"].to(device)
            inputs = {key: value.to(device) for key, value in batch.items() if key != "labels"}
            predicted = model(**inputs).logits.argmax(dim=-1)
            active = targets != -100
            correct += ((predicted == targets) & active).sum().item()
            total += active.sum().item()
            for row in predicted.cpu().tolist():
                offsets = spans[len(predictions)]
                predictions.append(decode_entities(row[:len(offsets)], offsets))
    metrics = evaluate_predictions(records, predictions)
    metrics["token_accuracy"] = correct / total if total else 0.0
    return metrics, predictions



def prepare_evaluation(records, tokenizer, config):
    encoded = [encode_record(record, tokenizer, config["max_length"]) for record in records]
    features = [inputs for inputs, _ in encoded]
    spans = [offsets for _, offsets in encoded]
    return make_loader(features, tokenizer, config), spans


def train_model(model, tokenizer, train_records, validation_records, config, device, checkpoint):
    if config["max_length"] > model.config.max_position_embeddings:
        raise ValueError("max_length превышает контекстное окно модели")
    train_features = [
        encode_record(record, tokenizer, config["max_length"])[0]
        for record in train_records
    ]
    train_loader = make_loader(train_features, tokenizer, config, shuffle=True)
    validation_loader, validation_spans = prepare_evaluation(validation_records, tokenizer, config)

    model.to(device)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"],
    )
    best_score, best_epoch = -1.0, None
    history = []
    for epoch in range(1, config["epochs"] + 1):
        model.train()
        loss_sum = token_count = 0
        for batch in train_loader:
            batch = {key: value.to(device) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            loss = model(**batch).loss
            if not torch.isfinite(loss):
                raise ValueError("Получена неконечная функция потерь")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["max_grad_norm"])
            optimizer.step()
            active_count = (batch["labels"] != -100).sum().item()
            loss_sum += loss.item() * active_count
            token_count += active_count
        metrics, _ = evaluate_model(
            model, validation_loader, validation_records, validation_spans, device,
        )
        score = metrics["micro"]["f1"]
        train_loss = loss_sum / token_count
        history.append({"epoch": epoch, "train_loss": train_loss, "validation_micro_f1": score})
        print(f"Эпоха {epoch}: loss={train_loss:.4f}, validation F1={score:.4f}")
        if score > best_score:
            best_score, best_epoch = score, epoch
            model.save_pretrained(checkpoint)
    tokenizer.save_pretrained(checkpoint)
    return history, best_epoch

In [ ]:
set_seed(config["seed"])
run_dir = Path(tempfile.mkdtemp(prefix="e1_rubert_", dir="/content"))
checkpoint = run_dir / "checkpoint"
tokenizer = AutoTokenizer.from_pretrained(
    config["model_id"], revision=config["revision"], use_fast=True, do_lower_case=False,
)
tokenizer.padding_side = "right"
model = AutoModelForTokenClassification.from_pretrained(
    config["model_id"], revision=config["revision"],
    num_labels=len(BIO_LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
    use_safetensors=False, attn_implementation="eager",
)
history, best_epoch = train_model(
    model, tokenizer, corpus["train"], corpus["validation"], config, device, checkpoint,
)
del model
print(f"Выбранная эпоха {best_epoch}")

## 5. Оценка лучшей модели

Обе оценки выполняются после загрузки сохранённой контрольной точки. На `test` считается итоговый результат; выбор эпохи уже завершён.

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    checkpoint, local_files_only=True, attn_implementation="eager",
).to(device)
tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True, use_fast=True)
evaluations = {}
for split in ("validation", "test"):
    records = corpus[split]
    loader, spans = prepare_evaluation(records, tokenizer, config)
    metrics, predictions = evaluate_model(model, loader, records, spans, device)
    evaluations[split] = (metrics, predictions)
    print(f"\n{split}, эпоха {best_epoch}")
    print(f"{'Класс':<14} {'Precision':>9} {'Recall':>9} {'F1':>9} {'FN':>5}")
    rows = {**metrics["per_class"], "micro": metrics["micro"], "macro": metrics["macro"]}
    for label, values in rows.items():
        print(f"{label:<14} {values['precision']:9.4f} {values['recall']:9.4f} "
              f"{values['f1']:9.4f} {values.get('fn', '—'):>5}")
    print(f"Точность по подтокенам {metrics['token_accuracy']:.4f}")

## 6. Сохранение результатов

Архив результатов содержит параметры, версии библиотек, оборудование, хеши входных файлов, журнал обучения, метрики обеих выборок и предсказанные символьные диапазоны. Предсказания сохраняются как идентификаторы записей и диапазоны без копирования текстов корпуса.

In [ ]:
results_dir = run_dir / "results"
results_dir.mkdir(exist_ok=True)
result = {
    "experiment": "E1-RuBERT",
    "training_mode": "full_fine_tuning",
    "selection_metric": "validation.micro.f1",
    "config": config,
    "label2id": LABEL2ID,
    "environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "tokenizers": tokenizers.__version__,
        "tokenizer_class": type(tokenizer).__name__,
        "device": str(device),
        "device_name": device_name,
        "cpu_threads": torch.get_num_threads(),
        "cuda": torch.version.cuda,
    },
    "input_sha256": input_hashes,
    "best_epoch": best_epoch,
    "history": history,
    "metrics": {split: metrics for split, (metrics, _) in evaluations.items()},
}
(results_dir / "e1-rubert-evaluation.json").write_text(
    json.dumps(result, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
)
for split, (_, predictions) in evaluations.items():
    path = results_dir / f"{split}-predictions.jsonl"
    with path.open("w", encoding="utf-8") as output:
        for record, entities in zip(corpus[split], predictions, strict=True):
            output.write(json.dumps(
                {"id": record["id"], "entities": entities}, ensure_ascii=False,
            ) + "\n")
archive = shutil.make_archive(str(run_dir / "e1_rubert_results"), "zip", results_dir)
files.download(archive)

### Скачать контрольную точку

Чтобы сохранить обученные веса и токенизатор для повторной оценки, установите `download_checkpoint = True` и запустите эту ячейку. Архив весит около 700 МБ.

In [ ]:
download_checkpoint = False

if download_checkpoint:
    checkpoint_archive = run_dir / "e1_rubert_checkpoint.zip"
    with zipfile.ZipFile(checkpoint_archive, "w") as archive:
        for path in sorted(checkpoint.iterdir()):
            if path.is_file():
                archive.write(path, path.name)
    files.download(str(checkpoint_archive))